# 55장 · create_agent의 도구 등록과 메시지 왕복

create_agent는 모델 호출과 도구 실행을 연결합니다. 대화 메시지에서 도구 요청과 결과를 찾아 어느 함수가 실행됐는지 확인합니다.

가상 도서관에서 주말 운영 시간과 대출 기간을 함께 물었습니다. 17장에서는 두 함수를 부르는 반복문을 직접 썼습니다. 여기서는 같은 질문을 받은 create_agent의 실행 기록을 읽고 모델이 요청한 일과 도구가 마친 일을 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain 1.4.6, langchain-core 1.6.12, langsmith 0.14.11입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 도구 요청과 결과 메시지 연결

도구 요청에는 함수 이름과 인자, 결과를 받을 때 사용할 호출 ID가 들어 있습니다. 시간을 알아내는 일은 등록된 함수가 맡습니다. 아래는 대본 모델이 낸 요청을 Python으로 직접 전달해 보는 예입니다.


### 셀 01 · 도구 함수의 인자와 결과

요청에 시간표 값이 있는지 먼저 보세요. 조회 뒤 어느 값이 새로 생길까요?


In [1]:
hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}
request = {"name": "get_hours", "args": {"day": "weekend"}, "id": "c1"}
def get_hours(day):
    return {"ok": True, "hours": hours[day]}

result = get_hours(**request["args"])
print(result["hours"])

10:00~16:00


10:00~16:00은 함수가 시간표에서 꺼낸 값입니다. 요청의 c1은 시간이 들어갈 칸이 아니라 도구 결과를 원래 요청과 짝지을 식별자입니다.


### 과제 02 · 요청과 결과의 ID 연결

두 번째 요청 c2의 응답에서 대출 기간을 꺼내세요. 목록 위치 대신 호출 ID를 쓰세요.


In [2]:
replies = {'c1': {'hours': '10:00~16:00'}, 'c2': {'loan_days': 14}}
# c2의 대출 기간을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>replies = {&#x27;c1&#x27;: {&#x27;hours&#x27;: &#x27;10:00~16:00&#x27;}, &#x27;c2&#x27;: {&#x27;loan_days&#x27;: 14}}
print(replies[&#x27;c2&#x27;][&#x27;loan_days&#x27;])</code></pre>
<p>실행 결과</p><pre><code>14</code></pre>
<p>14는 대출 규정 도구가 반환한 일수입니다. 반환 순서가 달라져도 c2로 찾아낼 수 있습니다.</p>
</details>


## 대화 기록에서 도구 호출 수 집계

프레임워크 예제 manual_conversion.py에서는 같은 두 함수를 create_agent에 등록하고 invoke로 질문을 넘깁니다. 도구 요청이 오면 프레임워크가 해당 함수를 부르고 결과를 돌려줍니다. 마지막 문장을 쓸 때도 모델이 한 번 더 응답합니다. 여기서는 기존 대본 모델 실행에서 기록한 순서를 읽습니다.


### 셀 03 · 도구 메시지 수 집계

대본에서 나온 메시지 종류입니다. tool이 등장한 수와 ai가 등장한 수를 따로 확인하세요.


In [3]:
roles = ["human", "ai", "tool", "ai", "tool", "ai"]
print(roles.count("tool"), roles.count("ai"))

2 3


2 3이 보입니다. 두 번의 도구 요청에 응답한 뒤 세 번째 모델 메시지가 최종 문장을 냈습니다. 이 기록은 고정 대본의 호출 횟수를 보여 주며 외부 모델 요금은 측정하지 않았습니다.


### 과제 04 · 대응 결과가 없는 호출

c1과 c2를 요청했지만 c1만 회신했습니다. 회신을 기다리는 ID를 출력하세요.


In [4]:
requested = ['c1', 'c2']
returned = {'c1'}
# 아직 결과가 없는 ID를 찾으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>requested = [&#x27;c1&#x27;, &#x27;c2&#x27;]
returned = {&#x27;c1&#x27;}
print([call_id for call_id in requested if call_id not in returned])</code></pre>
<p>실행 결과</p><pre><code>[&#x27;c2&#x27;]</code></pre>
<p>[&#x27;c2&#x27;]가 남습니다. 이 상태에서 대출 기간을 확정하면 도구가 주지 않은 답을 쓰게 됩니다.</p>
</details>


## create_agent에 두 도구 등록

이 절은 LangChain 1.x 수업 커널에서 실행합니다. 대본 모델은 질문의 의미를 판단하는 대신 정해 둔 순서대로 요청을 냅니다. 도구를 찾아 실행하고 그 결과를 메시지에 붙이는 부분은 실제 LangChain이 맡습니다.

### 셀 06에서 두 도구의 반환값은 어디로 갈까요?

**주말 시간표를 요청합니다.**　첫 모델 메시지에는 도구 이름과 요청 ID가 있습니다.

```
c1 · get_hours
```

**함수가 시간표를 돌려줍니다.**　도구 메시지의 tool_call_id는 첫 요청의 ID인 c1입니다.

```
c1 · 10:00~16:00
```

**대출 기간을 요청합니다.**　다음 모델 메시지가 두 번째 요청을 냅니다.

```
c2 · get_loan_rule
```

**함수가 대출 기간을 돌려줍니다.**　이 값은 c2가 요청한 대출 일수입니다.

```
c2 · 14
```

**마지막 모델 메시지가 답을 냅니다.**　추가 도구 요청이 없어 왕복이 끝납니다.

```
모델 메시지 3개 · 도구 메시지 2개
```


### 셀 05 · 도구 요청을 내는 테스트 모델

도구 요청을 담은 메시지를 만들 도우미입니다. 다음 셀에서 쓰므로 출력은 없습니다.


In [5]:
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langsmith import tracing_context

class ClassroomModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        return self  # 사용할 도구 요청은 아래 대본에 고정했습니다.

def ask_tool(name, args, call_id):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

bind_tools가 대본 모델을 그대로 돌려줍니다. 네트워크나 결제 계정 없이 같은 요청 순서를 반복해서 볼 수 있습니다.


### 셀 06 · create_agent의 도구 왕복

각 도구가 실제로 불렸을 때의 반환값을 호출 ID와 함께 읽습니다.

셀 01의 같은 이름 함수는 사전을 반환해서 hours 키를 읽었습니다. 여기에서 새로 정의하는 도구는 문자열을 돌려줍니다. 메시지에서 호출 ID를 먼저 찾은 뒤 그 ID와 짝을 이룬 ToolMessage의 content를 읽으세요.


In [6]:
def get_hours(day: str) -> str:
    """가상 도서관의 운영 시간을 찾는다."""
    return hours[day]

def get_loan_rule() -> str:
    """가상 도서관 대출 일수를 찾는다."""
    return "14"

model = ClassroomModel(responses=[
    ask_tool("get_hours", {"day": "weekend"}, "c1"),
    ask_tool("get_loan_rule", {}, "c2"),
    AIMessage(content="주말은 10:00~16:00이며 대출은 14일입니다."),
])
agent = create_agent(model, tools=[get_hours, get_loan_rule])
with tracing_context(enabled=False):
    state = agent.invoke({"messages": [("user", "주말 시간과 대출 기간은?")]})
for message in state["messages"]:
    if message.type == "tool":
        print(message.tool_call_id, message.content)

c1 10:00~16:00
c2 14


c1 뒤에 시간, c2 뒤에 14가 찍힙니다. 호출 ID와 내용은 실제로 돌려받은 ToolMessage에서 꺼냈습니다. 이 실습에서는 도구 요청과 마지막 답 문장을 대본에 미리 적었고, 함수의 반환값은 실행 중에 얻었습니다. 도구 결과의 tool_call_id에서 c1과 c2를 찾아 각 반환값을 연결하세요.


## 돌아보기

운영 시간 문장 하나를 골라 함수의 반환값과 요청 ID를 거슬러 올라가세요. 56장에서는 이 왕복의 어느 지점에서 잘못된 값을 막을지 정합니다.
